# Axial v3 Iteration B - low-cost runner

Orquestador B0-B6 reproducible. No ejecuta todos los experimentos por defecto.

In [3]:
# ============================================================
# NOTEBOOK 52 — PREPARACIÓN SEGURA PARA COLAB
# Ejecuta B0 en preflight por defecto.
# ============================================================

import importlib
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

# ------------------------------------------------------------
# 1. Dependencias
# ------------------------------------------------------------

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "PIL": "Pillow",
    "matplotlib": "matplotlib",
    "torch": "torch",
    "pydicom": "pydicom",
}

missing = [
    pip_package
    for module_name, pip_package
    in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        *missing,
    ])

import numpy as np
import pandas as pd
import pydicom
from PIL import Image


# ------------------------------------------------------------
# 2. Google Drive
# ------------------------------------------------------------

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if not IN_COLAB:
    raise RuntimeError(
        "Este notebook está preparado para Google Colab."
    )

from google.colab import drive

DRIVE_ROOT = Path("/content/drive")
MY_DRIVE = DRIVE_ROOT / "MyDrive"
PFI_ROOT = MY_DRIVE / "PFI_MVP"

if not MY_DRIVE.exists():
    drive.mount(
        str(DRIVE_ROOT),
        force_remount=False,
    )
else:
    print("Google Drive ya está montado.")

if not PFI_ROOT.is_dir():
    raise FileNotFoundError(
        f"No existe el directorio: {PFI_ROOT}"
    )


# ------------------------------------------------------------
# 3. Panel de control
# ------------------------------------------------------------

# DEJAR ASÍ PARA LA PRIMERA EJECUCIÓN
RUN_MODE_VALUE = "preflight"
EXPERIMENT_ID_VALUE = "B0"
AUTHORIZE_EXECUTION = False

# Más adelante:
#
# Smoke:
# RUN_MODE_VALUE = "smoke"
# EXPERIMENT_ID_VALUE = "B0"
# AUTHORIZE_EXECUTION = True
#
# Entrenamiento completo:
# RUN_MODE_VALUE = "train"
# EXPERIMENT_ID_VALUE = "B0"
# AUTHORIZE_EXECUTION = True
#
# Por ahora no usar train.


# ------------------------------------------------------------
# 4. Commit y rutas
# ------------------------------------------------------------

REPO_REF = (
    "2b8b943df73bd4a2f45ba4865158301d2d3889cf"
)

REPO_ROOT = Path(
    "/content/PFI_MVPTest_Enzo_AImodule"
)

REPO_URL = (
    "https://github.com/EnzoAA004/"
    "PFI_MVPTest_Enzo_AImodule.git"
)

MANIFEST_PATH = (
    PFI_ROOT
    / "results"
    / "E9_alkafri_axial_t2_final_labels_baseline"
    / "E9_t2_final_labels_curated_split.csv"
)

OUTPUT_BASE = PFI_ROOT / "outputs"

REGISTRY_PATH = (
    OUTPUT_BASE
    / "axial_v3"
    / "iteration_b"
    / "experiment_registry.csv"
)


# ------------------------------------------------------------
# 5. Variables de entorno
# Deben definirse antes de importar axial_v3.
# ------------------------------------------------------------

os.environ["PFI_REPO_REF"] = REPO_REF
os.environ["PFI_REPO_ROOT"] = str(REPO_ROOT)
os.environ["PFI_REPO_URL"] = REPO_URL

os.environ["PFI_DATASET_ROOT"] = str(PFI_ROOT)
os.environ["AXIAL_E9_CURATED_SPLIT_CSV"] = str(
    MANIFEST_PATH
)

os.environ["PFI_OUTPUT_ROOT"] = str(
    OUTPUT_BASE
)

os.environ["PFI_RESUME_ROOT"] = str(
    OUTPUT_BASE
)

os.environ["PFI_REGISTRY_PATH"] = str(
    REGISTRY_PATH
)

os.environ["RUN_MODE"] = RUN_MODE_VALUE

os.environ[
    "PFI_AXIAL_V3_EXPERIMENT_ID"
] = EXPERIMENT_ID_VALUE

os.environ[
    "PFI_AXIAL_V3_EXPERIMENT_TYPE"
] = EXPERIMENT_ID_VALUE.split("-")[0]

os.environ[
    "PFI_RUN_AXIAL_V3_EXPERIMENT"
] = "1" if AUTHORIZE_EXECUTION else "0"

os.environ["PFI_MASK_LABEL_MODE"] = "raw"
os.environ["PFI_NUM_WORKERS"] = "0"
os.environ["PFI_BATCH_SIZE"] = "8"
os.environ["PFI_USE_AMP"] = "1"
os.environ["RESUME_MODE"] = "auto"


# ------------------------------------------------------------
# 6. Validar archivos principales
# ------------------------------------------------------------

print({
    "manifestExists": MANIFEST_PATH.is_file(),
    "manifest": str(MANIFEST_PATH),
    "outputBase": str(OUTPUT_BASE),
    "registry": str(REGISTRY_PATH),
    "runMode": RUN_MODE_VALUE,
    "experimentId": EXPERIMENT_ID_VALUE,
    "authorized": AUTHORIZE_EXECUTION,
})

if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f"No se encontró el manifest: {MANIFEST_PATH}"
    )


# ------------------------------------------------------------
# 7. Clonar o actualizar repositorio
# ------------------------------------------------------------

if not (REPO_ROOT / ".git").exists():
    subprocess.check_call([
        "git",
        "clone",
        REPO_URL,
        str(REPO_ROOT),
    ])

subprocess.check_call([
    "git",
    "-C",
    str(REPO_ROOT),
    "fetch",
    "--all",
    "--tags",
])

subprocess.check_call([
    "git",
    "-C",
    str(REPO_ROOT),
    "checkout",
    REPO_REF,
])

effective_sha = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO_ROOT),
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

if effective_sha != REPO_REF:
    raise RuntimeError(
        "Commit efectivo inesperado: "
        f"{effective_sha} != {REPO_REF}"
    )

SRC_ROOT = REPO_ROOT / "src"

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_ROOT),
    )

print({
    "repoRoot": str(REPO_ROOT),
    "effectiveSha": effective_sha,
    "srcRoot": str(SRC_ROOT),
})


# ------------------------------------------------------------
# 8. Importar training y aplicar soporte médico
# ------------------------------------------------------------

import lumbar_mri.axial_v3.training as training

training = importlib.reload(training)

STANDARD_IMAGE_SUFFIXES = {
    ".png",
    ".jpg",
    ".jpeg",
    ".bmp",
    ".tif",
    ".tiff",
}


def open_2d_array_medical(
    path: Path,
) -> np.ndarray:
    """
    Loader compatible con axial-final-v2.

    Soporta:
    - .npy
    - imágenes convencionales
    - DICOM .ima y .dcm
    """

    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".npy":
        array = np.asarray(
            np.load(path)
        )

    elif suffix in STANDARD_IMAGE_SUFFIXES:
        array = np.asarray(
            Image.open(path)
        )

    elif suffix in {".ima", ".dcm"}:
        dicom = pydicom.dcmread(
            str(path)
        )

        array = np.asarray(
            dicom.pixel_array
        )

    else:
        raise ValueError(
            "Formato médico no soportado: "
            f"{suffix} ({path})"
        )

    array = np.asarray(array)

    if array.ndim == 2:
        return array

    if array.ndim == 3 and 1 in array.shape:
        squeezed = np.squeeze(array)

        if squeezed.ndim == 2:
            return squeezed

    if (
        suffix in STANDARD_IMAGE_SUFFIXES
        and array.ndim == 3
        and array.shape[-1] in {3, 4}
    ):
        return array[..., 0]

    raise ValueError(
        f"Shape 2D no soportado: "
        f"{array.shape} ({path})"
    )


# El dataset de training consulta este símbolo global.
training.open_2d_array = open_2d_array_medical

print({
    "trainingLoader": (
        training.open_2d_array.__name__
    ),
    "loaderModule": (
        training.open_2d_array.__module__
    ),
})


# ------------------------------------------------------------
# 9. Verificar manifest y probar un DICOM real
# ------------------------------------------------------------

manifest_df = pd.read_csv(
    MANIFEST_PATH
)

split_counts = (
    manifest_df["split"]
    .astype(str)
    .str.lower()
    .value_counts()
    .to_dict()
)

if split_counts != {
    "train": 427,
    "test": 102,
    "val": 81,
}:
    raise RuntimeError(
        "Distribución inesperada del manifest: "
        f"{split_counts}"
    )

development_df = manifest_df[
    manifest_df["split"]
    .astype(str)
    .str.lower()
    .isin(["train", "val", "validation"])
]

sample_dicom_path = None

for raw_path in development_df[
    "image_file_path"
].astype(str):

    path = Path(raw_path)

    if not path.is_absolute():
        path = PFI_ROOT / path

    if path.suffix.lower() in {".ima", ".dcm"}:
        sample_dicom_path = path
        break

if sample_dicom_path is None:
    raise RuntimeError(
        "No se encontró un DICOM de desarrollo."
    )

sample_array = training.open_2d_array(
    sample_dicom_path
)

print({
    "manifestRows": len(manifest_df),
    "splitCounts": split_counts,
    "developmentRows": len(development_df),
    "sampleDicom": str(sample_dicom_path),
    "sampleShape": list(sample_array.shape),
    "sampleDtype": str(sample_array.dtype),
    "sampleFinite": bool(
        np.isfinite(sample_array).all()
    ),
})

if sample_array.ndim != 2:
    raise RuntimeError(
        "El DICOM de prueba no quedó como array 2D."
    )

Mounted at /content/drive
{'manifestExists': True, 'manifest': '/content/drive/MyDrive/PFI_MVP/results/E9_alkafri_axial_t2_final_labels_baseline/E9_t2_final_labels_curated_split.csv', 'outputBase': '/content/drive/MyDrive/PFI_MVP/outputs', 'registry': '/content/drive/MyDrive/PFI_MVP/outputs/axial_v3/iteration_b/experiment_registry.csv', 'runMode': 'preflight', 'experimentId': 'B0', 'authorized': False}
{'repoRoot': '/content/PFI_MVPTest_Enzo_AImodule', 'effectiveSha': '2b8b943df73bd4a2f45ba4865158301d2d3889cf', 'srcRoot': '/content/PFI_MVPTest_Enzo_AImodule/src'}
{'trainingLoader': 'open_2d_array_medical', 'loaderModule': '__main__'}
{'manifestRows': 610, 'splitCounts': {'train': 427, 'test': 102, 'val': 81}, 'developmentRows': 508, 'sampleDicom': '/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T2_TSE_TRA_384_0004/T2_TSE_TRA__0001_003.ima', 'sampleShape': [320, 320], 'sampleDtype': 'uint16',

In [4]:
# ============================================================
# NOTEBOOK 52 — LOW-COST RUNNER
# ============================================================

from dataclasses import replace
from pathlib import Path

import json
import os

from lumbar_mri.axial_v3.experiments import (
    estimate_run_count,
    load_low_cost_grid,
    select_experiment,
)

from lumbar_mri.axial_v3.guards import (
    require_train_val_only,
)

# Importar desde el módulo que ya fue parcheado
AxialV3TrainConfig = (
    training.AxialV3TrainConfig
)

run_calibration = (
    training.run_calibration
)

run_preflight = (
    training.run_preflight
)

run_training = (
    training.run_training
)

summarize_validation_runs = (
    training.summarize_validation_runs
)


# ------------------------------------------------------------
# Seguridad train/validation
# ------------------------------------------------------------

ALLOWED_SPLITS = require_train_val_only(
    ["train", "val"],
    context="notebook_52",
)


# ------------------------------------------------------------
# Grid de experimentos
# ------------------------------------------------------------

CONFIG_PATH = (
    REPO_ROOT
    / "config"
    / "axial_v3_low_cost_grid.json"
)

if not CONFIG_PATH.is_file():
    raise FileNotFoundError(
        f"No existe el grid: {CONFIG_PATH}"
    )

GRID = load_low_cost_grid(
    CONFIG_PATH
)

RUN_MODE = os.getenv(
    "RUN_MODE",
    "preflight",
).strip().lower()

EXPERIMENT_ID = os.getenv(
    "PFI_AXIAL_V3_EXPERIMENT_ID",
    "B0",
).strip()

EXPLICIT_EXPERIMENT_ID = (
    "PFI_AXIAL_V3_EXPERIMENT_ID"
    in os.environ
)


# ------------------------------------------------------------
# Construcción de configuración
# ------------------------------------------------------------

def train_config_for_experiment(
    experiment_id: str,
):
    base = AxialV3TrainConfig.from_env()

    experiment = select_experiment(
        GRID,
        experiment_id,
    )

    return replace(
        base,

        RUN_ID=experiment.runId,

        EXPERIMENT_ID=(
            experiment.experimentId
        ),

        EXPERIMENT_TYPE=(
            experiment.experimentType
        ),

        RAW0_WEIGHT_BOOST=(
            experiment
            .raw0EffectiveWeightMultiplier
        ),

        MAX_CLASS_WEIGHT_RATIO=(
            experiment.maxClassWeightRatio
        ),

        LOSS_NAME=(
            experiment.lossName
        ),

        TVERSKY_ALPHA=(
            experiment.tverskyAlpha
            if experiment.tverskyAlpha
            is not None
            else base.TVERSKY_ALPHA
        ),

        TVERSKY_BETA=(
            experiment.tverskyBeta
            if experiment.tverskyBeta
            is not None
            else base.TVERSKY_BETA
        ),

        FOCAL_GAMMA=(
            experiment.focalGamma
            if experiment.focalGamma
            is not None
            else base.FOCAL_GAMMA
        ),

        RAW0_TVERSKY_WEIGHT=(
            experiment.raw0TverskyWeight
        ),

        RAW0_FP_PENALTY_WEIGHT=(
            experiment.raw0FpPenaltyWeight
        ),

        MIN_PROBABILITY=(
            experiment.minProbability
        ),

        MIN_MARGIN=(
            experiment.minMargin
        ),

        MAX_EPOCHS=(
            experiment.maxEpochs
        ),

        EARLY_STOPPING_PATIENCE=(
            experiment
            .earlyStoppingPatience
        ),

        PRESENCE_HEAD_ENABLED=(
            experiment.presenceHeadEnabled
        ),

        LAMBDA_PRESENCE=(
            experiment.lambdaPresence
        ),

        RAW0_BALANCED_SAMPLER_ENABLED=(
            experiment
            .raw0BalancedSamplerEnabled
        ),

        POSITIVE_FRACTION=(
            experiment.positiveFraction
            if experiment.positiveFraction
            is not None
            else 0.5
        ),

        PARENT_EXPERIMENT_ID=(
            experiment.parentExperimentId
        ),

        PARENT_RUN_ID=(
            experiment.parentRunId
        ),
    )


CFG = train_config_for_experiment(
    EXPERIMENT_ID
)

RUN_EXPERIMENT = (
    os.getenv(
        "PFI_RUN_AXIAL_V3_EXPERIMENT",
        "0",
    )
    == "1"
)


# ------------------------------------------------------------
# Mostrar configuración efectiva
# ------------------------------------------------------------

print(json.dumps(
    {
        "runMode": RUN_MODE,
        "experimentId": EXPERIMENT_ID,
        "experimentType": (
            CFG.EXPERIMENT_TYPE
        ),
        "runId": CFG.RUN_ID,
        "manifest": str(
            CFG.SPLIT_MANIFEST_PATH
        ),
        "datasetRoot": str(
            CFG.DATASET_ROOT
        ),
        "outputRoot": str(
            CFG.OUTPUT_ROOT
        ),
        "resumeRoot": str(
            CFG.RESUME_ROOT
        ),
        "registryPath": str(
            CFG.REGISTRY_PATH
        ),
        "authorized": RUN_EXPERIMENT,
        "plannedRuns": (
            estimate_run_count(GRID)
        ),
        "lossName": CFG.LOSS_NAME,
        "presenceHead": (
            CFG.PRESENCE_HEAD_ENABLED
        ),
        "balancedSampler": (
            CFG
            .RAW0_BALANCED_SAMPLER_ENABLED
        ),
    },
    indent=2,
    default=str,
))


# ------------------------------------------------------------
# Ejecutar modo seleccionado
# ------------------------------------------------------------

if RUN_MODE == "preflight":

    report = run_preflight(
        CFG,
        select_experiment(
            GRID,
            EXPERIMENT_ID,
        ),
    )

    print(json.dumps(
        {
            "mode": RUN_MODE,
            "experimentId": (
                EXPERIMENT_ID
            ),
            "plannedRuns": (
                estimate_run_count(GRID)
            ),
            "preflight": report,
        },
        indent=2,
        default=str,
    ))


elif RUN_MODE in {"smoke", "train"}:

    if (
        not EXPLICIT_EXPERIMENT_ID
        or not RUN_EXPERIMENT
    ):
        raise RuntimeError(
            "smoke/train requieren "
            "PFI_AXIAL_V3_EXPERIMENT_ID "
            "y autorización explícita."
        )

    if CFG.EXPERIMENT_TYPE == "B4":
        raise RuntimeError(
            "B4 es calibración. "
            "Usar RUN_MODE=calibrate."
        )

    if (
        RUN_MODE == "train"
        and not __import__("torch").cuda.is_available()
    ):
        raise RuntimeError(
            "RUN_MODE=train requiere GPU CUDA."
        )

    result = run_training(
        CFG,
        smoke=(RUN_MODE == "smoke"),
    )

    print(json.dumps(
        result,
        indent=2,
        default=str,
    ))


elif RUN_MODE == "calibrate":

    if (
        not EXPLICIT_EXPERIMENT_ID
        or not RUN_EXPERIMENT
    ):
        raise RuntimeError(
            "calibrate requiere "
            "experimento y autorización."
        )

    parent_raw = os.getenv(
        "PFI_AXIAL_V3_PARENT_CHECKPOINT",
        "",
    ).strip()

    if not parent_raw:
        raise RuntimeError(
            "Falta definir "
            "PFI_AXIAL_V3_PARENT_CHECKPOINT."
        )

    parent = Path(parent_raw)

    if not parent.is_file():
        raise FileNotFoundError(parent)

    result = run_calibration(
        CFG,
        parent_checkpoint_path=parent,
    )

    print(json.dumps(
        result,
        indent=2,
        default=str,
    ))


elif RUN_MODE == "summarize":

    result = summarize_validation_runs(
        CFG
    )

    print(json.dumps(
        result,
        indent=2,
        default=str,
    ))


else:

    raise ValueError(
        f"RUN_MODE inválido: {RUN_MODE}"
    )

{
  "runMode": "preflight",
  "experimentId": "B0",
  "experimentType": "B0",
  "runId": "axial-v3-B0",
  "manifest": "/content/drive/MyDrive/PFI_MVP/results/E9_alkafri_axial_t2_final_labels_baseline/E9_t2_final_labels_curated_split.csv",
  "datasetRoot": "/content/drive/MyDrive/PFI_MVP",
  "outputRoot": "/content/drive/MyDrive/PFI_MVP/outputs/axial_v3/iteration_b",
  "resumeRoot": "/content/drive/MyDrive/PFI_MVP/outputs/axial_v3/iteration_b/resume",
  "registryPath": "/content/drive/MyDrive/PFI_MVP/outputs/axial_v3/iteration_b/experiment_registry.csv",
  "authorized": false,
  "plannedRuns": 30,
  "lossName": "baseline_v2",
  "presenceHead": false,
  "balancedSampler": false
}
{
  "mode": "preflight",
  "experimentId": "B0",
  "plannedRuns": 30,
  "preflight": {
    "status": "preflight_passed",
    "experimentId": "B0",
    "runId": "axial-v3-B0",
    "trainSlices": 427,
    "valSlices": 81,
    "trainPatients": 128,
    "valPatients": 27,
    "raw0PresenceTrain": null,
    "classWei